# PP Model-PP水分不限模型

Read the root `README-交接说明.md` first. / 请先阅读根目录交接说明。

The same six crops, weather and interface are used in both modes. / 两种模式使用同一套六作物、天气与接口。
Storage-organ dry matter is reported. No nitrogen leaching is calculated. / 输出收获器官干物质，本版不计算氮淋失。


In [ ]:
from pathlib import Path
import runpy
import sys
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '06_Hao_Interface_and_Rerun-Hao接口与复跑').is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter inside the full handoff folder. / 请在完整交接文件夹内启动 Jupyter。")
registry = runpy.run_path(str(ROOT / '06_Hao_Interface_and_Rerun-Hao接口与复跑/Shared_Core-共享核心/Module_Loader-模块加载.py'))
registry["install"]()
from crop_engine import ModelConfig, GroundGrid, crop_catalog, prepare_weather, evaluate
registry["load_module"]("radiation_bridge")
from radiation_bridge import preflight, get_defaults, run_tianze_daily
print("Project folder / 项目目录:", ROOT)
print("Radiance status / 辐射环境:", preflight())


## 1. Crop and assumptions-作物与假设
六种作物都可以运行，也可以将 `CROP_IDS` 改成一个 ID。价格和参考产量不会覆盖模型计算。

In [ ]:
MODE = "PP"
ACCEPT_DEMO_ASSUMPTIONS = False
CROP_IDS = list(crop_catalog())
GRID = GroundGrid()
display(pd.DataFrame(crop_catalog()).T[["label", "variety", "start_mm_dd", "end_mm_dd", "organ", "reference_open_field_yield_t_ha"]])


## 2. Run natural-light baseline-运行自然光照对照
每种作物独立初始化；冬小麦跨年，其余为同年。天气来自同一 TMY 与配套液态降雨。

In [ ]:
natural_results = {}
natural_folders = {}
if ACCEPT_DEMO_ASSUMPTIONS:
    for crop_id in CROP_IDS:
        config = ModelConfig(crop_id=crop_id, mode=MODE, accept_demo_assumptions=True)
        result, folder = evaluate(config, grid=GRID)
        natural_results[crop_id] = result
        natural_folders[crop_id] = folder
    display(pd.DataFrame([{k:r[k] for k in ["crop_id","mode","yield_kg_DM_ha","status","transpiration_mm","drainage_bottom_mm"]} for r in natural_results.values()]))
else:
    print("尚未运行。阅读说明后，将 ACCEPT_DEMO_ASSUMPTIONS 改为 True。")


## 3. Inspect model results-检查模型结果
WLP 的层间水流和底部排水有明确深度；PP 不生成动态分层水分。

In [ ]:
if natural_folders:
    crop_id = CROP_IDS[0]
    folder = natural_folders[crop_id]
    display(pd.read_csv(folder / "Point_Summary-采样点汇总.csv"))
    if (folder / "Daily_Soil_Layers-逐日分层土壤.csv").exists():
        display(pd.read_csv(folder / "Daily_Soil_Layers-逐日分层土壤.csv").head(20))
    else:
        display(pd.read_csv(folder / "Daily_Crop_Growth-逐日作物生长.csv").head())
    print("精确输出目录：", folder)


## 4. Connect actual radiation-接入真实辐射
选择对应作物完整季节的文件。保持 `GRID` 与 Tianze 输出网格一致。缺失日期会被拒绝，不把参考 LER 当作辐射。

In [ ]:
RADIATION_CSV = None  # 真实逐日×逐点辐射 CSV 路径
RADIATION_PROVENANCE = None  # 真实计算来源和 run_id
SCENARIO_ID = "fixed_conventional"
SELECTED_CROP = CROP_IDS[0]
if RADIATION_CSV is not None:
    if not ACCEPT_DEMO_ASSUMPTIONS or not RADIATION_PROVENANCE:
        raise ValueError("请先确认演示假设并填写真实辐射来源。")
    config = ModelConfig(crop_id=SELECTED_CROP, mode=MODE, accept_demo_assumptions=True)
    result, folder = evaluate(config, radiation=RADIATION_CSV, scenario_id=SCENARIO_ID,
        provenance=RADIATION_PROVENANCE, grid=GRID, baseline=natural_results.get(SELECTED_CROP))
    display(pd.DataFrame([result]).drop(columns=["reference_table_row"], errors="ignore"))
    print(folder)
else:
    print("尚未提供真实 APV 辐射，没有生成虚构 APV 产量。")


## 5. Optional Tianze radiation-可选Tianze辐射计算
这一步不会默认运行，可能耗时很长。先做单日核查，再将日期换成完整作物季。

In [ ]:
RUN_SINGLE_DAY_RADIANCE = False
if RUN_SINGLE_DAY_RADIANCE:
    config = ModelConfig(crop_id=SELECTED_CROP, mode=MODE, accept_demo_assumptions=True)
    weather, weather_info = prepare_weather(config)
    defaults = get_defaults()["fixed"]
    radiation, source = run_tianze_daily("fixed", [weather.day.iloc[0]], GRID,
        design=defaults["design"], module_spec=defaults["module"], array_spec=defaults["array"],
        albedo=0.20, epw_path=ROOT / '03_Precipitation-降水条件' / "Aachen_TMY_Weather-Aachen典型年天气.epw", allow_run=True,
        accuracy="high")
    display(radiation.head())
    print(source)
